# Bag-of-Words vs TF-IDF — which one should classical ML use?

Both turn text into numeric vectors classical models (Logistic Regression, SVM, ...) can use. This
notebook settles which one to standardize on, using an experiment instead of assuming.

**What it does:**
1. Load the shared train/val/test splits (saved by `data_eda.ipynb`)
2. Train the *same* Logistic Regression on BoW vectors vs TF-IDF vectors, compare on val F1
3. Whichever wins, tune ITS key parameters (`max_features`, `ngram_range`) to find the best config
4. Every run is logged to MLflow so the conclusion is backed by numbers, not a guess

Run `data_eda.ipynb` first — this notebook depends on the CSVs it saves.

In [1]:
# ── libraries ──
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

import mlflow

/Users/vijay/Desktop/sentiment-drl-trading/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── config + load the shared splits ──
DATA_DIR = "../data/processed"
SEED = 42
MLFLOW_EXPERIMENT = "IndianFinSentiment_ClassicalML"

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment(MLFLOW_EXPERIMENT)

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
val_df = pd.read_csv(f"{DATA_DIR}/val.csv")
print(f"Train: {len(train_df)}  |  Val: {len(val_df)}")


def evaluate(y_true, y_pred):
    return {
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "f1_weighted": round(f1_score(y_true, y_pred, average="weighted"), 4),
    }

Train: 7094  |  Val: 946


## Part 1 — Bag-of-Words vs TF-IDF (same classifier, same data, only the vectorizer changes)

In [3]:
def train_and_log(vectorizer, run_name):
    X_train = vectorizer.fit_transform(train_df["text"])
    X_val = vectorizer.transform(val_df["text"])

    model = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    model.fit(X_train, train_df["label_id"])
    preds = model.predict(X_val)
    metrics = evaluate(val_df["label_id"], preds)

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({"vectorizer": run_name, "max_features": 15000, "ngram_range": "(1,2)"})
        mlflow.log_metrics(metrics)

    print(f"{run_name:<20} -> {metrics}")
    return metrics


bow_metrics = train_and_log(
    CountVectorizer(max_features=15_000, ngram_range=(1, 2), min_df=2), "BoW"
)
tfidf_metrics = train_and_log(
    TfidfVectorizer(max_features=15_000, ngram_range=(1, 2), min_df=2), "TF-IDF"
)

winner = "TF-IDF" if tfidf_metrics["f1_weighted"] >= bow_metrics["f1_weighted"] else "BoW"
print(f"\nWinner: {winner} (BoW F1={bow_metrics['f1_weighted']:.4f}, TF-IDF F1={tfidf_metrics['f1_weighted']:.4f})")
print(f"-> Every classical-ML notebook after this one uses {winner}.")

BoW                  -> {'accuracy': 0.8097, 'f1_weighted': 0.8096}
TF-IDF               -> {'accuracy': 0.8087, 'f1_weighted': 0.8083}

Winner: BoW (BoW F1=0.8096, TF-IDF F1=0.8083)
-> Every classical-ML notebook after this one uses BoW.


## Part 2 — Now tune the winner's parameters

We just picked a vectorizer above — now find its best `max_features` / `ngram_range` combination,
instead of keeping the arbitrary `max_features=15000, ngram_range=(1,2)` default used above. Small
grid, each combination logged as its own MLflow run.

In [4]:
VectorizerClass = TfidfVectorizer if winner == "TF-IDF" else CountVectorizer

results = []
for max_features in [5_000, 10_000, 15_000, 20_000, 30_000]:
    for ngram_range in [(1, 1), (1, 2), (1, 3)]:
        vectorizer = VectorizerClass(max_features=max_features, ngram_range=ngram_range, min_df=2)
        X_train = vectorizer.fit_transform(train_df["text"])
        X_val = vectorizer.transform(val_df["text"])

        model = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
        model.fit(X_train, train_df["label_id"])
        preds = model.predict(X_val)
        metrics = evaluate(val_df["label_id"], preds)

        run_name = f"{winner}_maxfeat{max_features}_ngram{ngram_range}"
        with mlflow.start_run(run_name=run_name):
            mlflow.log_params({"max_features": max_features, "ngram_range": str(ngram_range)})
            mlflow.log_metrics(metrics)

        results.append({"max_features": max_features, "ngram_range": ngram_range, **metrics})
        print(f"max_features={max_features:>6}  ngram_range={str(ngram_range):<8} -> f1={metrics['f1_weighted']:.4f}")

results_df = pd.DataFrame(results).sort_values("f1_weighted", ascending=False)
best = results_df.iloc[0]
print(f"\nBest config: max_features={best['max_features']}, ngram_range={best['ngram_range']}, f1={best['f1_weighted']:.4f}")
print(f"-> Use these two values as TFIDF_MAX_FEATURES / TFIDF_NGRAM_RANGE in the *_with_hpt.ipynb notebooks.")

max_features=  5000  ngram_range=(1, 1)   -> f1=0.8109
max_features=  5000  ngram_range=(1, 2)   -> f1=0.8043
max_features=  5000  ngram_range=(1, 3)   -> f1=0.8077
max_features= 10000  ngram_range=(1, 1)   -> f1=0.8109
max_features= 10000  ngram_range=(1, 2)   -> f1=0.8054
max_features= 10000  ngram_range=(1, 3)   -> f1=0.8117
max_features= 15000  ngram_range=(1, 1)   -> f1=0.8109
max_features= 15000  ngram_range=(1, 2)   -> f1=0.8096
max_features= 15000  ngram_range=(1, 3)   -> f1=0.8074
max_features= 20000  ngram_range=(1, 1)   -> f1=0.8109
max_features= 20000  ngram_range=(1, 2)   -> f1=0.8096
max_features= 20000  ngram_range=(1, 3)   -> f1=0.8085
max_features= 30000  ngram_range=(1, 1)   -> f1=0.8109
max_features= 30000  ngram_range=(1, 2)   -> f1=0.8096
max_features= 30000  ngram_range=(1, 3)   -> f1=0.8085

Best config: max_features=10000, ngram_range=(1, 3), f1=0.8117
-> Use these two values as TFIDF_MAX_FEATURES / TFIDF_NGRAM_RANGE in the *_with_hpt.ipynb notebooks.
